# كشف أمراض أوراق الذرة — خط أنابيب Colab الكامل

دفتر التشغيل الذي أُنتجت به نتائج التقرير (`report/corn_leaf_disease_report_ar.html`).

**ترتيب المراحل:** تنزيل البيانات → التحقّق منها → التنظيف (SHA-256 + تعارض الوسوم) → التقسيم → الرسوم → خطوط `tf.data` → بناء النموذج والتدريب بمرحلتين → التقييم → تصدير المقاييس → مخطط البنية → (اختياري) TFLite → التنبؤ.

> يعمل على Google Colab (يستخدم `google.colab.files` لتنزيل المخرجات). خلاياه مأخوذة كما نُفِّذت، ولم تُعدّل.

## 1) التثبيت والإعدادات العامة

In [ ]:
import sys, subprocess, os, json, random, shutil, zipfile, hashlib, platform
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub', 'scikit-learn', 'pandas', 'matplotlib', 'pillow'], check=True)
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score
from google.colab import files

SEED = 42
IMG_SIZE = (224, 224)
BATCH = 32
EPOCHS_HEAD = 10
EPOCHS_FINE = 5
OUT = Path('/content/corn_outputs')
OUT.mkdir(parents=True, exist_ok=True)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception as e:
    print('Determinism:', type(e).__name__)
print('TensorFlow:', tf.__version__, 'GPU:', tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    print('تنبيه: لا يوجد GPU؛ قد يستغرق التدريب وقتًا أطول.')

import kagglehub
print('تنزيل البيانات العامة؛ لا يلزم رفع kaggle.json عادةً.')
DATA_ROOT = Path(kagglehub.dataset_download(
    'smaranjitghose/corn-or-maize-leaf-disease-dataset'
))
print('اكتمل التحميل:', DATA_ROOT)

## 2) تحديد مجلد الفئات والتنظيف والتقسيم

In [ ]:
MANUAL_DATA_DIR = None  # مثال: '/content/corn_data/data'
EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif'}
def has_images(folder):
    return any(p.is_file() and p.suffix.lower() in EXTS for p in folder.rglob('*'))

candidates = []
for folder in [DATA_ROOT] + sorted(p for p in DATA_ROOT.rglob('*') if p.is_dir()):
    children = sorted(p for p in folder.iterdir() if p.is_dir() and not p.name.startswith('.'))
    if len(children) == 4 and all(has_images(p) for p in children):
        candidates.append(folder)
if MANUAL_DATA_DIR:
    DATA_DIR = Path(MANUAL_DATA_DIR)
elif len(candidates) == 1:
    DATA_DIR = candidates[0]
else:
    print('Candidates:', [str(p) for p in candidates])
    raise ValueError('حدد MANUAL_DATA_DIR ثم أعد تشغيل الخلية')

class_dirs = sorted(p for p in DATA_DIR.iterdir() if p.is_dir() and not p.name.startswith('.'))
class_names = [p.name for p in class_dirs]
assert len(class_names) == 4, class_names
print('DATA_DIR:', DATA_DIR)
print('Class order:', dict(enumerate(class_names)))
valid_rows, rejected = [], []
for label, folder in enumerate(class_dirs):
    for path in sorted(folder.rglob('*')):
        if not path.is_file() or path.suffix.lower() not in EXTS:
            continue
        try:
            with Image.open(path) as image:
                image.verify()
            digest = hashlib.sha256(path.read_bytes()).hexdigest()
        except Exception as error:
            rejected.append({'path': str(path), 'reason': str(error)})
            continue
        valid_rows.append({'path': str(path), 'label': label,
                           'class_name': folder.name, 'sha256': digest})
if not valid_rows:
    raise ValueError('لم يُعثر على صور صالحة')
all_images = pd.DataFrame(valid_rows)
labels_per_hash = all_images.groupby('sha256')['label'].nunique()
conflict_hashes = set(labels_per_hash[labels_per_hash > 1].index)
conflicting_images = all_images[all_images.sha256.isin(conflict_hashes)].copy()
non_conflicting = all_images[~all_images.sha256.isin(conflict_hashes)].copy()
duplicate_mask = non_conflicting.duplicated(subset='sha256', keep='first')
duplicates = non_conflicting[duplicate_mask].copy()
manifest = non_conflicting[~duplicate_mask].copy().reset_index(drop=True)
assert len(manifest) > 0, 'لا توجد صور بعد التنظيف'
assert set(manifest.label) == set(range(4)), 'إحدى الفئات أصبحت فارغة'
pd.DataFrame(rejected, columns=['path','reason']).to_csv(OUT/'rejected_images.csv', index=False)
conflicting_images.to_csv(OUT/'conflicting_labels.csv', index=False)
duplicates.to_csv(OUT/'exact_duplicates_removed.csv', index=False)
manifest.to_csv(OUT/'clean_manifest.csv', index=False)
print('Valid images before cleaning:', len(all_images))
print('Conflicting hash groups:', len(conflict_hashes))
print('All conflicting-label images excluded:', len(conflicting_images))
print('Additional exact duplicates removed:', len(duplicates))
train_df, val_df = train_test_split(manifest, test_size=0.2, random_state=SEED, shuffle=True)
train_df, val_df = train_df.copy(), val_df.copy()
assert set(train_df.sha256).isdisjoint(set(val_df.sha256))
assert set(train_df.label) == set(range(4)), 'إحدى الفئات غائبة عن التدريب'
assert set(val_df.label) == set(range(4)), 'إحدى الفئات غائبة عن التحقق'
train_df['split'], val_df['split'] = 'train', 'validation'
pd.concat([train_df, val_df]).to_csv(OUT/'split_manifest.csv', index=False)
counts = pd.DataFrame(index=class_names)
for name, frame in [('total', manifest), ('train', train_df), ('validation', val_df)]:
    counts[name] = frame.class_name.value_counts().reindex(class_names, fill_value=0)
counts.index.name = 'class_name'
counts.to_csv(OUT/'class_counts.csv')
display(counts)
print('Total / train / validation:', len(manifest), len(train_df), len(val_df))
print('Removed exact duplicates:', len(duplicates), 'Unreadable:', len(rejected))

## 3) رسوم توزيع الفئات وأمثلة الصور

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, name in zip(axes, class_names):
    p = manifest.loc[manifest.class_name == name, 'path'].iloc[0]
    with Image.open(p) as im:
        ax.imshow(im.convert('RGB'))
    ax.set_title(name, fontsize=10)
    ax.axis('off')
fig.suptitle('Corn leaf dataset examples (not model predictions)')
fig.tight_layout()
fig.savefig(OUT/'class_examples.png', dpi=220, bbox_inches='tight')
plt.show()
ax = counts[['train', 'validation']].plot.bar(stacked=True, figsize=(9, 5), color=['#20563f', '#c7a75f'])
ax.set(title='Class distribution after exact-duplicate removal', ylabel='Images', xlabel='Class')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(OUT/'class_distribution.png', dpi=220, bbox_inches='tight')
plt.show()

## 4) خطوط `tf.data`

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
def read_image(path, label):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(tf.cast(image, tf.float32), IMG_SIZE)
    return image, tf.cast(label, tf.int32)

def make_ds(frame, training=False):
    ds = tf.data.Dataset.from_tensor_slices((frame.path.to_numpy(), frame.label.to_numpy(dtype=np.int32)))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(read_image, num_parallel_calls=AUTOTUNE, deterministic=True)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

train_ds = make_ds(train_df, training=True)
val_ds = make_ds(val_df)

## 5) بناء النموذج والتدريب بمرحلتين

In [ ]:
from tensorflow.keras import layers, models
ROTATION_FACTOR = 0.10
augmentation = models.Sequential([
    layers.RandomFlip('horizontal', seed=SEED),
    layers.RandomRotation(ROTATION_FACTOR, seed=SEED+1),
    layers.RandomZoom(height_factor=0.10, width_factor=0.10, seed=SEED+2),
], name='augmentation')
base = tf.keras.applications.MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet')
base.trainable = False
inputs = layers.Input(shape=(*IMG_SIZE, 3), name='rgb_0_to_255')
x = augmentation(inputs)
x = layers.Rescaling(1/127.5, offset=-1, name='mobilenetv2_normalization')(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3, seed=SEED)(x)
outputs = layers.Dense(len(class_names), activation='softmax', name='class_scores')(x)
model = models.Model(inputs, outputs, name='corn_mobilenetv2')
with open(OUT/'model_summary.txt', 'w') as f:
    model.summary(print_fn=lambda line: f.write(line+'\n'))
model.summary()

def compile_model(lr):
    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
def callbacks(stage):
    return [
        tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=3, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(str(OUT/f'{stage}_best.keras'), monitor='val_accuracy', mode='max', save_best_only=True),
        tf.keras.callbacks.CSVLogger(str(OUT/f'{stage}_history.csv')),
        tf.keras.callbacks.TerminateOnNaN(),
    ]
compile_model(1e-3)
h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, callbacks=callbacks('stage1'))
# Ensure fine-tuning starts from the exact checkpoint selected in stage 1.
model.load_weights(OUT/'stage1_best.keras')
base.trainable = True
for layer in base.layers:
    layer.trainable = False
for layer in base.layers[-30:]:
    if not isinstance(layer, layers.BatchNormalization):
        layer.trainable = True
compile_model(1e-5)
h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINE, callbacks=callbacks('stage2'))
print('Actual epochs:', len(h1.history['loss']), len(h2.history['loss']))

## 6) اختيار النموذج ومنحنيات التدريب

In [ ]:
best1 = float(max(h1.history['val_accuracy']))
best2 = float(max(h2.history['val_accuracy']))
selected_stage = 'stage2' if best2 > best1 else 'stage1'
selected_model = tf.keras.models.load_model(OUT/f'{selected_stage}_best.keras')
selected_model.save(OUT/'corn_best.keras')
print('Selected:', selected_stage)
print('Stage 1 best validation accuracy:', best1)
print('Stage 2 best validation accuracy:', best2)
print('Change in percentage points:', (best2-best1)*100)

n1 = len(h1.history['loss'])
combined = {k: h1.history[k]+h2.history[k] for k in h1.history}
with open(OUT/'history.json', 'w') as f:
    json.dump({'stage1': h1.history, 'stage2': h2.history}, f, indent=2)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, title in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
    epochs = np.arange(1, len(combined[metric])+1)
    ax.plot(epochs, combined[metric], label='Train', color='#20563f')
    ax.plot(epochs, combined['val_'+metric], label='Validation', color='#b88b36')
    ax.axvline(n1+0.5, color='gray', linestyle='--', label='Fine-tuning starts')
    ax.set(title=title, xlabel='Epoch', ylabel=title)
    ax.legend(); ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUT/'training_curves.png', dpi=220, bbox_inches='tight')
plt.show()

## 7) التقييم على مجموعة التحقق وتوقّع كل صورة

In [ ]:
y_true, y_pred, probabilities = [], [], []
for images, labels in val_ds:
    probs = selected_model(images, training=False).numpy()
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(np.argmax(probs, axis=1).tolist())
    probabilities.extend(probs.tolist())
y_true, y_pred = np.array(y_true), np.array(y_pred)
probabilities = np.asarray(probabilities)
report = classification_report(y_true, y_pred, labels=list(range(4)), target_names=class_names,
                               output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).T
report_df.to_csv(OUT/'classification_report.csv')
print(classification_report(y_true, y_pred, labels=list(range(4)), target_names=class_names,
                            digits=4, zero_division=0))
cm = confusion_matrix(y_true, y_pred, labels=list(range(4)))
pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(OUT/'confusion_matrix_counts.csv')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, norm, title in [(axes[0], None, 'Validation: counts'), (axes[1], 'true', 'Validation: row-normalized')]:
    c = confusion_matrix(y_true, y_pred, labels=list(range(4)), normalize=norm)
    ConfusionMatrixDisplay(c, display_labels=class_names).plot(ax=ax, cmap='Greens', colorbar=False,
            xticks_rotation=25, values_format='d' if norm is None else '.2f')
    ax.set_title(title)
fig.tight_layout()
fig.savefig(OUT/'confusion_matrix.png', dpi=220, bbox_inches='tight')
plt.show()
predictions = val_df.reset_index(drop=True)[['path', 'class_name']].copy()
predictions['predicted_class'] = [class_names[i] for i in y_pred]
predictions['max_softmax_score'] = probabilities.max(axis=1)
for i, name in enumerate(class_names):
    predictions['score_'+name] = probabilities[:, i]
predictions.to_csv(OUT/'validation_predictions.csv', index=False)
with open(OUT/'class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)

## 8) تصدير المقاييس والملخّص العربي

In [ ]:
macro = report['macro avg']
accuracy = float(accuracy_score(y_true, y_pred))
model_mb = (OUT/'corn_best.keras').stat().st_size / (1024**2)
weakest = min(class_names, key=lambda name: report[name]['f1-score'])
offdiag = cm.copy()
np.fill_diagonal(offdiag, 0)
if offdiag.max() > 0:
    i, j = np.unravel_index(offdiag.argmax(), offdiag.shape)
    confusion_text = f'{class_names[i]} -> {class_names[j]} ({int(offdiag[i,j])} images)'
else:
    confusion_text = 'No validation misclassifications'
summary = {
    'validation_accuracy_percent': accuracy*100,
    'macro_precision': float(macro['precision']),
    'macro_recall': float(macro['recall']),
    'macro_f1': float(macro['f1-score']),
    'last_training_accuracy_percent': float(h2.history['accuracy'][-1])*100,
    'last_training_epoch_stage': 'stage2',
    'model_file_size_MiB': model_mb,
    'total_parameters': int(selected_model.count_params()),
    'selected_stage': selected_stage,
    'stage1_best_validation_accuracy_percent': best1*100,
    'stage2_best_validation_accuracy_percent': best2*100,
    'fine_tuning_change_percentage_points': (best2-best1)*100,
    'stage1_actual_epochs': len(h1.history['loss']),
    'stage2_actual_epochs': len(h2.history['loss']),
    'total_images_after_exact_dedup': len(manifest),
    'training_images': len(train_df), 'validation_images': len(val_df),
    'exact_duplicates_removed': len(duplicates),
    'conflicting_label_images_removed': len(conflicting_images),
    'conflicting_hash_groups': len(conflict_hashes),
    'valid_images_before_cleaning': len(all_images),
    'unreadable_images': len(rejected),
    'lowest_f1_class': weakest, 'largest_directional_confusion': confusion_text,
    'class_counts': counts.reset_index().to_dict(orient='records'),
    'per_class_metrics': {name: report[name] for name in class_names},
    'seed': SEED, 'split': 'random 80/20, not stratified, no independent test',
    'rotation_factor': ROTATION_FACTOR, 'rotation_degrees': ROTATION_FACTOR*360,
    'tensorflow_version': tf.__version__, 'python_version': platform.python_version(),
    'gpu': [d.name for d in tf.config.list_physical_devices('GPU')],
}
with open(OUT/'report_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
pd.DataFrame([(k,v) for k,v in summary.items() if not isinstance(v,(list,dict))],
             columns=['metric','value']).to_csv(OUT/'report_metrics.csv', index=False)
lines = [
    '# نتائج تصنيف أمراض أوراق الذرة',
    '',
    '**تنبيه:** هذه نتائج تحقق مستخدمة لاختيار النموذج، وليست اختبارًا مستقلاً.',
    '',
    f'- الصور بعد التنظيف وإزالة التكرار وتعارض الوسوم: {len(manifest)}؛ التدريب: {len(train_df)}؛ التحقق: {len(val_df)}.',
    f'- دقة التحقق للنموذج المختار: {accuracy*100:.2f}%.',
    f"- Macro Precision / Recall / F1: {macro['precision']:.4f} / {macro['recall']:.4f} / {macro['f1-score']:.4f}.",
    f"- دقة التدريب في آخر حقبة منفذة من المرحلة الثانية: {h2.history['accuracy'][-1]*100:.2f}%.",
    f'- أفضل مرحلة: {selected_stage}؛ حجم ملف Keras: {model_mb:.2f} MiB.',
    f'- تحسن أفضل دقة تحقق في المرحلة الثانية: {(best2-best1)*100:+.2f} نقطة مئوية.',
    f'- أقل فئة F1: {weakest}؛ قيمتها: {report[weakest]["f1-score"]:.4f}.',
    f'- أكبر خلط اتجاهي (عند التعادل يُعرض أول زوج): {confusion_text}.',
    '', f'- استُبعدت {len(conflicting_images)} صورة ضمن {len(conflict_hashes)} مجموعة متطابقة ذات وسوم متعارضة، و{len(duplicates)} نسخة إضافية متطابقة ضمن الفئة نفسها.',
    '## توزيع الفئات والنتائج',
    '| Class | Total | Train | Validation | Precision | Recall | F1 |',
    '|---|---:|---:|---:|---:|---:|---:|',
]
for name in class_names:
    r = report[name]; c = counts.loc[name]
    lines.append(f"| {name} | {c.total} | {c.train} | {c.validation} | {r['precision']:.4f} | {r['recall']:.4f} | {r['f1-score']:.4f} |")
lines += ['', '## مراجعة المناقشة',
          '- افحص منحنيات الدقة والخسارة قبل الاستنتاج بوجود فرط تعلّم.',
          '- دقة التدريب تُقاس مع التعزيز وDropout، لذلك لا تُقارن مباشرةً بالتحقق دون هذا التحفظ.',
          '- راجع علاقة عدد العينات بـRecall؛ الارتباط وحده لا يثبت السببية.',
          '- يلزم اختبار حقلي مستقل وفحص الصور المتشابهة بصريًا ومعايرة الثقة قبل الاستخدام العملي.',
          '', f'التدوير المستخدم: ±{ROTATION_FACTOR*360:.1f} درجة؛ التعزيز للتدريب فقط.']
(OUT/'report_summary_ar.md').write_text('\n'.join(lines), encoding='utf-8')
print('\n'.join(lines))

## 9) مخطط بنية النموذج و(اختياري) تصدير TFLite

In [ ]:
from matplotlib.patches import FancyBboxPatch
fig, ax = plt.subplots(figsize=(15, 3))
ax.set(xlim=(0, 15), ylim=(0, 3)); ax.axis('off')
labels = ['RGB input\n224 x 224 x 3', 'Train-only\naugmentation', 'Rescale\n[-1, 1]', 'MobileNetV2\nImageNet', 'Global Average\nPooling + Dropout 0.3', 'Dense (4)\nSoftmax']
for i, label in enumerate(labels):
    x = 0.15 + i*2.48
    ax.add_patch(FancyBboxPatch((x, 0.85), 2.15, 1.25, boxstyle='round,pad=0.04',
                              facecolor='#20563f' if i in (3,5) else '#edf3ee', edgecolor='#8aab97'))
    ax.text(x+1.075, 1.475, label, ha='center', va='center', fontsize=10,
            color='white' if i in (3,5) else '#203c2d')
    if i < 5:
        ax.annotate('', xy=(x+2.43, 1.475), xytext=(x+2.17, 1.475), arrowprops={'arrowstyle':'->'})
fig.savefig(OUT/'model_architecture.png', dpi=220, bbox_inches='tight')
plt.show()

EXPORT_TFLITE = False  # فعّلها إذا أردت تجربة التصدير
if EXPORT_TFLITE:
    try:
        export_dir = OUT/'saved_model_for_tflite'
        spec = [tf.TensorSpec([None, *IMG_SIZE, 3], tf.float32, name='images')]
        @tf.function(input_signature=spec)
        def serving(images):
            return {'scores': selected_model(images, training=False)}
        tf.saved_model.save(selected_model, str(export_dir), signatures={'serving_default': serving})
        converter = tf.lite.TFLiteConverter.from_saved_model(str(export_dir))
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        tflite_bytes = converter.convert()
        (OUT/'corn_model.tflite').write_bytes(tflite_bytes)
        print('TFLite file size (MiB):', len(tflite_bytes)/(1024**2))
        shutil.rmtree(export_dir)
    except Exception as e:
        (OUT/'tflite_export_error.txt').write_text(str(e))
        print('تعذر تصدير TFLite؛ بقية النتائج محفوظة:', type(e).__name__)

## 10) تجميع المخرجات وتنزيلها

In [ ]:
archive_path = shutil.make_archive('/content/corn_report_outputs', 'zip', root_dir=str(OUT))
print('Files:', sorted(p.name for p in OUT.iterdir()))
files.download(archive_path)

## 11) تنبؤ بصورة واحدة (اختياري)

In [ ]:
# مثال اختياري؛ لا تنفذه إلا بعد تعيين مسار صورة.
IMAGE_PATH = None
if IMAGE_PATH:
    loaded = tf.keras.models.load_model(OUT/'corn_best.keras')
    names = json.loads((OUT/'class_names.json').read_text())
    with Image.open(IMAGE_PATH) as im:
        image = np.asarray(im.convert('RGB'), dtype=np.float32)
    image = tf.image.resize(image, IMG_SIZE)
    scores = loaded(image[None, ...], training=False).numpy()[0]
    print('Predicted class:', names[int(np.argmax(scores))])
    print('Softmax score (not calibrated confidence):', float(scores.max()))